# 12 — 3D workflow from CSV calibration

Clean core workflow:

```text
matching 3D-image CSV
→ extract channel–mass calibration
→ build total channel spectrum from raw layers
→ calibrated Spectrum
→ peak assignment
→ mass bins
→ channel bins
→ SIMSVolume
```

This notebook intentionally avoids advanced plotting/export. Those are separated into notebooks 13–15.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re

import pandas as pd
import plotly.io as pio

from pymagsims import Spectrum, SIMSVolume
from pymagsims.raw_image import SIMSRawImage
from pymagsims.isotopes import load_builtin_isotopes, filter_isotopes
from pymagsims.binning import mass_bins_to_channel_bins
from pymagsims.interactive import plot_spectrum_with_peaks_interactive

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go


# Firefox/JupyterLab often works better with iframe.
pio.renderers.default = "iframe"

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"
CSV_3D_FILE = DATA / "202505077-MEMS-011_neg_500mT_3dimage.csv"

IMAGE_SHAPE = (256, 256)

## 1. Load raw image layer paths

In [ ]:
def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(RAW_LAYER_DIR.glob("*Image_*.raw"), key=natural_sort_key)

print(f"Found {len(paths)} raw image layers")
for p in paths:
    print(p.name)

## 2. Extract channel–mass calibration from the matching CSV

Recommended package cleanup: move `extract_calibration_from_3d_csv()` into `src/pymagsims/io.py`.

In [ ]:
def extract_calibration_from_3d_csv(path, encoding="latin1", n_channels=12000):
    path = Path(path)

    with path.open("r", encoding=encoding) as f:
        lines = f.readlines()

    end_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("%end"):
            end_idx = i
            break

    if end_idx is None:
        raise ValueError("Could not find %end marker in CSV file.")

    spectrum_rows = []
    spectrum_start = end_idx + 1

    for line in lines[spectrum_start:spectrum_start + n_channels]:
        parts = [p.strip() for p in line.strip().split(";") if p.strip() != ""]
        if len(parts) >= 3:
            spectrum_rows.append(parts[:3])

    spectrum_df = pd.DataFrame(
        spectrum_rows,
        columns=["Channel", "Mass", "Amplitude"],
    )
    spectrum_df = spectrum_df.apply(pd.to_numeric, errors="coerce").dropna()
    spectrum_df["Channel"] = spectrum_df["Channel"].astype(int)

    calibration_df = spectrum_df[["Channel", "Mass"]].copy()
    return spectrum_df, calibration_df

In [ ]:
csv_spectrum_df, calibration_df = extract_calibration_from_3d_csv(CSV_3D_FILE)

display(csv_spectrum_df.head())
display(calibration_df.head())
display(calibration_df.tail())

print("Channel range:", calibration_df["Channel"].min(), "to", calibration_df["Channel"].max())
print("Mass range:", calibration_df["Mass"].min(), "to", calibration_df["Mass"].max())

## 3. Save calibration for reuse

In [ ]:
calibration_dir = DATA / "calibration"
calibration_dir.mkdir(parents=True, exist_ok=True)

calibration_file = calibration_dir / "channel_mass_calibration_from_3d_csv.csv"
calibration_df.to_csv(calibration_file, index=False)

print(f"Saved calibration to: {calibration_file}")

## 4. Build total channel spectrum from raw image layers

In [ ]:
all_counts = []

for path in paths:
    raw = SIMSRawImage.from_fpd_raw(path, shape=IMAGE_SHAPE)
    counts = raw.events["Channel"].value_counts()
    all_counts.append(counts)

channel_counts = (
    pd.concat(all_counts, axis=1)
    .fillna(0)
    .sum(axis=1)
    .sort_index()
)

channel_spectrum = pd.DataFrame(
    {
        "Channel": channel_counts.index.astype(int),
        "Counts": channel_counts.values.astype(int),
    }
)

display(channel_spectrum.head())
print("Total counts:", channel_spectrum["Counts"].sum())

## 5. Construct calibrated Spectrum

In [ ]:
calibrated_spectrum_df = calibration_df.merge(
    channel_spectrum.rename(columns={"Counts": "Amplitude"}),
    on="Channel",
    how="left",
)

calibrated_spectrum_df["Amplitude"] = calibrated_spectrum_df["Amplitude"].fillna(0)

spec = Spectrum(
    data=calibrated_spectrum_df[["Channel", "Mass", "Amplitude"]],
    metadata={
        "source": "3D raw image layers with calibration extracted from matching CSV",
        "calibration_file": str(calibration_file),
    },
    name="3d_raw_image_calibrated_spectrum",
)

spec.plot(x="Mass", y="Amplitude", log_y=True);

## 6. Peak assignment

In [ ]:
isotopes = load_builtin_isotopes(max_atomic_number=79)
isotopes = filter_isotopes(isotopes, min_abundance=0.5)

assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
)

assignments = assignments.sort_values(["measured_mass", "abs_mass_error"])
display(assignments.head(30))
print(f"Number of candidate assignments: {len(assignments)}")

## 7. Interactive spectrum with peaks

In [ ]:
fig, assignments = plot_spectrum_with_peaks_interactive(
    spec,
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
    log_y=True,
    label_peaks=True,
)

fig.show()

## 8. Create mass bins and convert to channel bins

In [ ]:
mass_bins = spec.create_bins_from_assignments(assignments, width=0.3)

bin_dir = DATA / "bins"
bin_dir.mkdir(parents=True, exist_ok=True)

mass_bin_file = bin_dir / "mass_bins_from_3d_csv_calibration.csv"
mass_bins.to_csv(mass_bin_file, index=False)

converted_bins = mass_bins_to_channel_bins(mass_bins, calibration_df)

channel_bin_file = bin_dir / "channel_bins_from_3d_csv_calibration.csv"
converted_bins.to_csv(channel_bin_file, index=False)

display(converted_bins.head(20))

print(f"Saved mass bins to: {mass_bin_file}")
print(f"Saved channel bins to: {channel_bin_file}")

## 9. Build SIMSVolume

Keep individual isotope/peak bins separate. Do not merge channel ranges by element before reconstruction.

In [ ]:
selected_bins = converted_bins.copy()

volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=selected_bins,
    spectrum=None,
    include_total=True,
    shape=IMAGE_SHAPE,
)

print(volume.labels()[:20])
display(volume.metadata)

for label, arr in list(volume.volumes.items())[:10]:
    print(f"{label}: shape={arr.shape}, total_counts={arr.sum()}")

## 10. Save lightweight workflow state

The actual 3D arrays are not saved here. Rebuild them from raw data or export in notebook 15.

In [ ]:
summary_file = DATA / "bins" / "3d_workflow_summary.txt"

with summary_file.open("w") as f:
    f.write(f"CSV_3D_FILE={CSV_3D_FILE}\n")
    f.write(f"RAW_LAYER_DIR={RAW_LAYER_DIR}\n")
    f.write(f"IMAGE_SHAPE={IMAGE_SHAPE}\n")
    f.write(f"N_RAW_LAYERS={len(paths)}\n")
    f.write(f"MASS_BIN_FILE={mass_bin_file}\n")
    f.write(f"CHANNEL_BIN_FILE={channel_bin_file}\n")

print(f"Saved summary to: {summary_file}")